In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score

from Feature_extraction.feature_extractor import FeatureExtractor
from Embeddings.Models.embedding_generator import EmbeddingGenerator
from Embeddings.Semantics.descriptions import get_descriptions
from Model.model import IP_SAE_MODEL

In [2]:
DATASET_PATH  = r"../Data/UiS4ADL/Processed/UiS4ADL_100hz.csv"
ADL_DICT_PATH = '../Data/adl_dict.json'
RESULTS_DIR   = "../Results"

FS             = 100
WINDOW_SECONDS = 4.0
OVERLAP_RATIO  = 0.0
METHOD         = 'temporal_frequency'

EMBEDDING_MODEL  = 'all-MiniLM-L6-v2'
DESCRIPTION_TYPE = 'original_fadi'
USE_PROMPT       = True
LAMBDA           = 0.001

UNSEEN_CLASSES = [17, 14, 11, 9, 24, 5]     # fixed official split
N_RUNS_INFERENCE = 35                       # stochastic iterations per predict call
N_REPEATS        = 50                       # how many times to call predict_zsl
RANDOM_SEED      = 22
np.random.seed(RANDOM_SEED)

In [3]:
data = pd.read_csv(DATASET_PATH)
print(f"Loaded dataset: {data.shape}")
all_classes = sorted(np.unique(data['adl']))

with open(ADL_DICT_PATH, 'r') as f:
    adl_dict_raw = json.load(f)

# Load embeddings
embedding_generator = EmbeddingGenerator(output_dir="../Data/Embeddings")
embeddings = embedding_generator.load_embeddings(
    model_name=EMBEDDING_MODEL,
    desc_type=DESCRIPTION_TYPE,
    use_prompt=USE_PROMPT,
    dataset='fadi'
)
activity_descriptions = get_descriptions(DESCRIPTION_TYPE)
activity_labels       = sorted(activity_descriptions.keys())
class_to_embedding_idx = {label: idx for idx, label in enumerate(activity_labels)}

# Extract features
feature_extractor = FeatureExtractor()
sensor_cols = [c for c in data.columns if c not in ['timestamp', 'adl', 'session', 'subject', 'fileID']]

X, y, _, _ = feature_extractor.extract_features(
    data=data,
    method=METHOD,
    sensor_columns=sensor_cols,
    window_seconds=WINDOW_SECONDS,
    overlap_ratio=OVERLAP_RATIO,
    fs=FS,
    strategy="retain_short"
)
y = np.array(y)

# Build fixed train/test split
seen_classes   = [cls for cls in all_classes if cls not in UNSEEN_CLASSES]
seen_mask      = np.isin(y, seen_classes)
unseen_mask    = np.isin(y, UNSEEN_CLASSES)

X_train, y_train = X[seen_mask], y[seen_mask]
X_test,  y_test  = X[unseen_mask], y[unseen_mask]

print(f"\nFixed split — seen: {len(seen_classes)}, unseen: {len(UNSEEN_CLASSES)}")
print(f"Train samples: {len(y_train)}, Test samples: {len(y_test)}")

# Build indexed embedding matrix
max_class_id = max(all_classes)
all_embeddings_indexed = np.zeros((max_class_id + 1, embeddings.shape[1]))
for cls, idx in class_to_embedding_idx.items():
    if cls in all_classes:
        all_embeddings_indexed[cls] = embeddings[idx]

unseen_indices    = [class_to_embedding_idx[cls] for cls in UNSEEN_CLASSES]
unseen_embeddings = embeddings[unseen_indices]

Loaded dataset: (8424664, 28)
Loaded embeddings from: ../Data/Embeddings/all_minilm_l6_v2_original_fadi_prompt_fadi.npz
Model: all-MiniLM-L6-v2
Shape: (24, 384)

Extracting features: temporal_frequency
  Window: 4.0s
  Overlap: 0.0 (400 stride)
  Sampling rate: 100 Hz

Creating windows per fileID...
  Found 1445 files that are too short (30.94% of the available files)
  Created 20103 windows

Extracting features using method: temporal_frequency...
  Processing window 0/20103
                    1000/20103
                    2000/20103
                    3000/20103
                    4000/20103
                    5000/20103
                    6000/20103
                    7000/20103
                    8000/20103
                    9000/20103
                    10000/20103
                    11000/20103
                    12000/20103
                    13000/20103
                    14000/20103
                    15000/20103
                    16000/20103
                 

In [ ]:
model = IP_SAE_MODEL(lambda_reg=LAMBDA, scale_features=True)
model.fit(X_train, y_train, all_embeddings_indexed, verbose=0)

# Run inference N_REPEATS times with non-fixed seeds
print(f"\nRunning inference {N_REPEATS} times (fixed_seed=False, n_runs={N_RUNS_INFERENCE})...")
accuracies = []

for i in range(N_REPEATS):
    y_pred = model.predict_zsl(
        X_test,
        unseen_embeddings,
        UNSEEN_CLASSES,
        n_runs=N_RUNS_INFERENCE,
        fixed_seed=False          # source of variance
    )
    acc = balanced_accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    print(f"  Run {i+1:>2}/{N_REPEATS}: BA = {acc:.4%}")

accuracies = np.array(accuracies)

mean_acc = accuracies.mean() * 100
std_acc  = accuracies.std()  * 100
min_acc  = accuracies.min()  * 100
max_acc  = accuracies.max()  * 100
range_pp = max_acc - min_acc

print(f"Inference Stability Results (K={len(UNSEEN_CLASSES)}, fixed split)")
print(f"{'='*45}")
print(f"Mean BA  : {mean_acc:.4f}%")
print(f"Std      : {std_acc:.4f} pp")
print(f"Min      : {min_acc:.4f}%")
print(f"Max      : {max_acc:.4f}%")
print(f"Range    : {range_pp:.4f} pp")


Running inference 50 times (fixed_seed=False, n_runs=35)...
  Run  1/50: BA = 71.4107%
  Run  2/50: BA = 71.4716%
  Run  3/50: BA = 71.3592%
  Run  4/50: BA = 71.3953%
  Run  5/50: BA = 71.4656%
  Run  6/50: BA = 71.4993%
  Run  7/50: BA = 71.3452%
  Run  8/50: BA = 71.2958%
  Run  9/50: BA = 71.4986%
  Run 10/50: BA = 71.4257%
  Run 11/50: BA = 71.4446%
  Run 12/50: BA = 71.4302%
  Run 13/50: BA = 71.4832%
  Run 14/50: BA = 71.4176%
  Run 15/50: BA = 71.3840%
  Run 16/50: BA = 71.4154%
  Run 17/50: BA = 71.4498%
  Run 18/50: BA = 71.3479%
  Run 19/50: BA = 71.3325%
  Run 20/50: BA = 71.4169%
  Run 21/50: BA = 71.2716%
  Run 22/50: BA = 71.3445%
  Run 23/50: BA = 71.4028%
  Run 24/50: BA = 71.4317%
  Run 25/50: BA = 71.4615%
  Run 26/50: BA = 71.2968%
  Run 27/50: BA = 71.3200%
  Run 28/50: BA = 71.3102%
  Run 29/50: BA = 71.4478%
  Run 30/50: BA = 71.5347%
  Run 31/50: BA = 71.3297%
  Run 32/50: BA = 71.4232%
  Run 33/50: BA = 71.2842%
  Run 34/50: BA = 71.4201%
  Run 35/50: BA = 71.